In [3]:
# ============================================================
# FINAL MULTI-CANCER MULTI-OMICS PIPELINE
# Aggregated Analysis + Publication Quality Plots + Head Ablation + CSV Export
# ============================================================
!pip install captum xgboost
import os
import json
import re
import random
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from collections import defaultdict

from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

# ============================================================
# CONFIG & STYLE
# ============================================================

SEED = 42
MAX_EPOCHS = 500
MIN_EPOCHS = 150 
PATIENCE = 45
LR = 1e-3
WEIGHT_DECAY = 5e-4

# Logic & Loss Weights
ALIGN_W = 0.2    
ORTHO_W = 0.1       
GATE_ENT_W = 0.05   
SPARSITY_W = 0.01   

OMICS_DROPOUT_P = 0.15
TOP_K_GENES = 20
DEFAULT_N_SPLITS = 5
LATENT_DIM = 128

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)

BASE_DIR = "./preprocessing/processed_multicancer"
RESULTS_ROOT = "./results_aggregated"
os.makedirs(RESULTS_ROOT, exist_ok=True)

GS_TYPES = ["GS-BRCA", "GS-LGG", "GS-OV", "GS-COAD", "GS-GBM"]
OMICS = ["mRNA", "miRNA", "CNV", "Methy"]

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
warnings.filterwarnings("ignore")

# ============================================================
# 1. LOSS FUNCTIONS
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, smoothing=0.1):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.smoothing = smoothing

    def forward(self, logits, targets):
        n = logits.size(1)
        with torch.no_grad():
            true_dist = torch.zeros_like(logits)
            true_dist.fill_(self.smoothing / (n - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1 - self.smoothing)
        log_p = F.log_softmax(logits, dim=1)
        p = torch.exp(log_p)
        focal = (1 - p) ** self.gamma
        loss = -true_dist * focal * log_p
        if self.alpha is not None:
            loss = loss * self.alpha
        return loss.sum(dim=1).mean()

def alignment_loss(zs):
    loss = 0
    for i in range(len(zs)):
        for j in range(i + 1, len(zs)):
            zi = F.normalize(zs[i], dim=1)
            zj = F.normalize(zs[j], dim=1)
            loss += 1 - (zi * zj).sum(dim=1).mean()
    return loss

def orthogonality_loss(zs):
    loss = 0
    for i in range(len(zs)):
        for j in range(i + 1, len(zs)):
            zi = F.normalize(zs[i], dim=1)
            zj = F.normalize(zs[j], dim=1)
            loss += torch.abs((zi * zj).sum(dim=1)).mean()
    return loss

def gate_entropy(g):
    eps = 1e-6
    return -(g * torch.log(g + eps) + (1 - g) * torch.log(1 - g + eps)).mean()

# ============================================================
# 2. MODEL ARCHITECTURE
# ============================================================

class OmicsEncoder(nn.Module):
    def __init__(self, in_dim, latent_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.4),
            nn.Linear(256, latent_dim),
            nn.ReLU()
        )
    def forward(self, x): return self.net(x)

class ContextGate(nn.Module):
    def __init__(self, context_dim, gate_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(context_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, gate_dim),
            nn.Sigmoid()
        )
    def forward(self, context): return self.fc(context)

class GatedMultiOmicsClassifier(nn.Module):
    def __init__(self, in_dims, num_classes):
        super().__init__()
        self.encoders = nn.ModuleDict({k: OmicsEncoder(v, LATENT_DIM) for k, v in in_dims.items()})
        total_latent_dim = LATENT_DIM * len(in_dims)
        self.gates = nn.ModuleDict({k: ContextGate(total_latent_dim, LATENT_DIM) for k in in_dims})
        
        self.classifier = nn.Sequential(
            nn.Linear(total_latent_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def extract_features(self, x_dict):
        zs_map = {m: self.encoders[m](x_dict[m]) for m in self.encoders}
        global_context = torch.cat(list(zs_map.values()), dim=1) 
        gated_list = [zs_map[m] * self.gates[m](global_context) for m in self.encoders]
        return torch.cat(gated_list, dim=1)

    def forward(self, x_dict):
        zs_map = {m: self.encoders[m](x_dict[m]) for m in self.encoders}
        global_context = torch.cat(list(zs_map.values()), dim=1) 
        
        zs_list = []
        gated_list = []
        gates_map = {}

        for m in self.encoders:
            z = zs_map[m]
            g = self.gates[m](global_context)
            gated_list.append(z * g)
            zs_list.append(z)
            gates_map[m] = g

        fused = torch.cat(gated_list, dim=1)
        logits = self.classifier(fused)
        return logits, zs_list, gates_map

# ============================================================
# 3. TRAINING & COLLECTION LOGIC
# ============================================================

def train_and_collect(omics_data, y, tr, te, cancer_name, fold_idx, accumulators):
    Xtr, Xte = {}, {}
    for m, X in omics_data.items():
        sc = StandardScaler()
        Xtr[m] = sc.fit_transform(X[tr])
        Xte[m] = sc.transform(X[te])

    ytr, yte = y[tr], y[te]
    Xtr_t = {m: torch.tensor(v, dtype=torch.float32).to(DEVICE) for m, v in Xtr.items()}
    Xte_t = {m: torch.tensor(v, dtype=torch.float32).to(DEVICE) for m, v in Xte.items()}
    ytr_t = torch.tensor(ytr, dtype=torch.long).to(DEVICE)

    w = compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)
    w = np.power(w, 1.25)
    w = torch.tensor(w, dtype=torch.float32).to(DEVICE)
    
    current_gamma = 3.5 if cancer_name == "GS-COAD" else 2.0
    loss_fn = FocalLoss(alpha=w, gamma=current_gamma, smoothing=0.05)
    
    in_dims = {m: Xtr[m].shape[1] for m in Xtr}
    model = GatedMultiOmicsClassifier(in_dims, len(np.unique(y))).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_prec, wait = 0, 0
    best_state = None

    for epoch in range(MAX_EPOCHS):
        model.train()
        opt.zero_grad()
        
        train_inputs = {k: v.clone() for k,v in Xtr_t.items()}
        if torch.rand(1).item() < OMICS_DROPOUT_P:
            drop_omics = np.random.choice(list(train_inputs.keys()))
            train_inputs[drop_omics] = torch.zeros_like(train_inputs[drop_omics])

        logits, zs, gates = model(train_inputs)
        
        l_focal = loss_fn(logits, ytr_t) 
        l_align = ALIGN_W * alignment_loss(zs)
        l_ortho = ORTHO_W * orthogonality_loss(zs)
        l_ent = GATE_ENT_W * sum(gate_entropy(g) for g in gates.values())
        l_sparse = SPARSITY_W * sum(g.mean() for g in gates.values())

        loss = l_focal + l_align + l_ortho + l_ent + l_sparse
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            preds = model(Xte_t)[0].argmax(1).cpu().numpy()
            prec = precision_score(yte, preds, average="weighted", zero_division=0)

        if prec > best_prec:
            best_prec = prec
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if epoch >= MIN_EPOCHS and wait >= PATIENCE:
            break

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    # ========================================================
    # DATA COLLECTION & CLASSIFIER HEAD ABLATION
    # ========================================================
    model.eval()
    with torch.no_grad():
        preds_base = model(Xte_t)[0].argmax(1).cpu().numpy()
        
        base_prec = best_prec
        base_rec = recall_score(yte, preds_base, average="weighted", zero_division=0)
        base_f1 = f1_score(yte, preds_base, average="weighted", zero_division=0)
        
        accumulators['ablation']['Base_MLP']['prec'].append(base_prec)
        accumulators['ablation']['Base_MLP']['rec'].append(base_rec)
        accumulators['ablation']['Base_MLP']['f1'].append(base_f1)
        
        accumulators['y_true'].extend(yte)
        accumulators['y_pred'].extend(preds_base)
        
        _, _, gate_vals = model(Xte_t)
        for m in OMICS: accumulators['gates'][m].append(gate_vals[m].mean().item())

        fused_tr = model.extract_features(Xtr_t).cpu().numpy()
        fused_te = model.extract_features(Xte_t).cpu().numpy()

    svm = SVC(kernel='rbf', class_weight='balanced', random_state=SEED)
    svm.fit(fused_tr, ytr)
    preds_svm = svm.predict(fused_te)
    accumulators['ablation']['SVM']['prec'].append(precision_score(yte, preds_svm, average="weighted", zero_division=0))
    accumulators['ablation']['SVM']['rec'].append(recall_score(yte, preds_svm, average="weighted", zero_division=0))
    accumulators['ablation']['SVM']['f1'].append(f1_score(yte, preds_svm, average="weighted", zero_division=0))

    le = LabelEncoder()
    ytr_xgb = le.fit_transform(ytr)
    yte_xgb = le.transform(yte)
    xgb = XGBClassifier(eval_metric='mlogloss', random_state=SEED, n_estimators=100)
    xgb.fit(fused_tr, ytr_xgb)
    preds_xgb = le.inverse_transform(xgb.predict(fused_te))
    accumulators['ablation']['XGBoost']['prec'].append(precision_score(yte, preds_xgb, average="weighted", zero_division=0))
    accumulators['ablation']['XGBoost']['rec'].append(recall_score(yte, preds_xgb, average="weighted", zero_division=0))
    accumulators['ablation']['XGBoost']['f1'].append(f1_score(yte, preds_xgb, average="weighted", zero_division=0))

    mlp2 = MLPClassifier(hidden_layer_sizes=(256, 128, 64), max_iter=500, random_state=SEED)
    mlp2.fit(fused_tr, ytr)
    preds_mlp2 = mlp2.predict(fused_te)
    accumulators['ablation']['Deeper_MLP']['prec'].append(precision_score(yte, preds_mlp2, average="weighted", zero_division=0))
    accumulators['ablation']['Deeper_MLP']['rec'].append(recall_score(yte, preds_mlp2, average="weighted", zero_division=0))
    accumulators['ablation']['Deeper_MLP']['f1'].append(f1_score(yte, preds_mlp2, average="weighted", zero_division=0))

    for m in OMICS:
        inputs = Xte_t[m].clone().requires_grad_(True)
        full_inputs = {k: Xte_t[k] for k in OMICS}
        full_inputs[m] = inputs 
        
        logits, _, _ = model(full_inputs)
        score = logits.max(dim=1)[0].mean()
        score.backward()
        
        sens = inputs.grad.abs().mean(dim=0).cpu().numpy()
        if accumulators['sensitivity'][m] is None:
            accumulators['sensitivity'][m] = np.zeros_like(sens)
        if accumulators['sensitivity'][m].shape == sens.shape:
             accumulators['sensitivity'][m] += sens

# ============================================================
# 4. FINAL AGGREGATED VISUALIZATION
# ============================================================

def generate_aggregated_plots(cancer_name, accumulators, n_folds, feature_names):
    save_dir = os.path.join(RESULTS_ROOT, cancer_name)
    os.makedirs(save_dir, exist_ok=True)

    # 1. Gate Importance Plot (Resized to be compact)
    gate_means = {m: np.mean(vals) for m, vals in accumulators['gates'].items()}
    gate_stds = {m: np.std(vals) for m, vals in accumulators['gates'].items()}
    
    plt.figure(figsize=(4.5, 3.5))
    x_vals, y_vals, y_err = list(gate_means.keys()), list(gate_means.values()), list(gate_stds.values())
    sns.barplot(x=x_vals, y=y_vals, palette="viridis", capsize=0.1)
    plt.errorbar(x=x_vals, y=y_vals, yerr=y_err, fmt='none', c='black', capsize=5)
    plt.title(f"{cancer_name}: Gate Importance", fontweight='bold')
    plt.ylabel("Mean Gate Activation")
    plt.ylim(0, 1.1)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "aggregated_gate_importance.png"))
    plt.close()

    # 2. Top 20 Features Plot (Resized to fit nicely in a paper column)
    for m in OMICS:
        if accumulators['sensitivity'][m] is not None:
            avg_sens = accumulators['sensitivity'][m] / n_folds
            
            full_idx = np.argsort(avg_sens)[::-1]
            full_scores = avg_sens[full_idx]
            full_names = [feature_names[m][i] for i in full_idx]

            idx_top20 = full_idx[:TOP_K_GENES]
            scores_top20 = full_scores[:TOP_K_GENES]
            names_top20 = full_names[:TOP_K_GENES]
            
            plt.figure(figsize=(4.5, 5.5))
            sns.barplot(x=scores_top20, y=names_top20, palette="magma", orient='h')
            plt.title(f"{cancer_name} ({m}): Top 20", fontweight='bold')
            plt.xlabel("Sensitivity Score")
            plt.tight_layout()
            plt.savefig(os.path.join(save_dir, f"aggregated_top20_{m}.png"))
            plt.close()

# ============================================================
# 5. MAIN LOOP
# ============================================================

if __name__ == "__main__":
    global_ablation_summary = []

    for CANCER in GS_TYPES:
        print(f"\n==================== {CANCER} ====================")
        DATA_DIR = os.path.join(BASE_DIR, CANCER)
        
        try:
            omics_data = {
                "mRNA":  np.load(os.path.join(DATA_DIR, "mRNA_processed.npy")),
                "miRNA": np.load(os.path.join(DATA_DIR, "miRNA_processed.npy")),
                "CNV":   np.load(os.path.join(DATA_DIR, "CNV_processed.npy")),
                "Methy": np.load(os.path.join(DATA_DIR, "Methy_processed.npy")),
            }
            y = np.load(os.path.join(DATA_DIR, "labels.npy"))
        except FileNotFoundError:
            print(f"Skipping {CANCER} (Files not found)")
            continue
            
        feature_names = {}
        for m in OMICS:
            num_feats = omics_data[m].shape[1]
            json_path = os.path.join(DATA_DIR, f"{m}_features.json")
            feats = []
            
            if os.path.exists(json_path):
                with open(json_path, 'r', encoding='utf-8') as f:
                    content = f.read().strip()
                try:
                    j_data = json.loads(content)
                    if isinstance(j_data, dict):
                        sorted_items = sorted(j_data.items(), key=lambda x: int(x[0]) if str(x[0]).isdigit() else x[0])
                        feats = [v for k, v in sorted_items]
                    else:
                        feats = j_data
                except json.JSONDecodeError:
                    feats = re.findall(r'"([^"]+)"', content)

                if len(feats) == num_feats:
                    feature_names[m] = feats
                elif len(feats) > num_feats:
                    feature_names[m] = feats[:num_feats]
                elif len(feats) > 0:
                    padding = [f"{m}_Feat_{i}" for i in range(len(feats), num_feats)]
                    feature_names[m] = feats + padding
                else:
                    feature_names[m] = [f"{m}_Feat_{i}" for i in range(num_feats)]
            else:
                feature_names[m] = [f"{m}_Feat_{i}" for i in range(num_feats)]

        accumulators = {
            'y_true': [],
            'y_pred': [],
            'gates': defaultdict(list),
            'sensitivity': {m: None for m in OMICS},
            'ablation': {
                'Base_MLP': {'prec': [], 'rec': [], 'f1': []},
                'SVM': {'prec': [], 'rec': [], 'f1': []},
                'XGBoost': {'prec': [], 'rec': [], 'f1': []},
                'Deeper_MLP': {'prec': [], 'rec': [], 'f1': []}
            }
        }

        min_class_size = np.min(np.unique(y, return_counts=True)[1])
        n_splits = max(2, min_class_size) if min_class_size < DEFAULT_N_SPLITS else DEFAULT_N_SPLITS
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

        for i, (tr, te) in enumerate(skf.split(omics_data["mRNA"], y)):
            print(f"  > Fold {i}...")
            train_and_collect(omics_data, y, tr, te, CANCER, i, accumulators)
            plt.close('all')

        print(f"\n{CANCER} Classifier Ablation Results (Averaged across folds):")
        for clf, metrics in accumulators['ablation'].items():
            mean_prec = np.mean(metrics['prec'])
            mean_rec  = np.mean(metrics['rec'])
            mean_f1   = np.mean(metrics['f1'])
            print(f"  > {clf:12s} | Prec: {mean_prec:.4f} | Rec: {mean_rec:.4f} | F1: {mean_f1:.4f}")
            
            # Save only the means to the global summary
            global_ablation_summary.append({
                'Cancer': CANCER,
                'Classifier': clf,
                'Mean_Precision': mean_prec,
                'Mean_Recall': mean_rec,
                'Mean_F1': mean_f1
            })
        
        print(f"  > Generating aggregated plots for {CANCER}...")
        generate_aggregated_plots(CANCER, accumulators, n_splits, feature_names)

    # Output ONLY the single master ablation summary CSV
    if global_ablation_summary:
        master_df = pd.DataFrame(global_ablation_summary)
        master_csv_path = os.path.join(RESULTS_ROOT, "final_ablation_summary_all_cancers.csv")
        master_df.to_csv(master_csv_path, index=False)
        print(f"\nALL PROCESSES COMPLETED. Master summary saved to: {master_csv_path}")
    else:
        print("\nALL PROCESSES COMPLETED.")


==================== GS-BRCA ====================
  > Fold 0...
  > Fold 1...
  > Fold 2...
  > Fold 3...
  > Fold 4...

GS-BRCA Classifier Ablation Results (Averaged across folds):
  > Base_MLP     | Prec: 0.8901 | Rec: 0.8793 | F1: 0.8815
  > SVM          | Prec: 0.8566 | Rec: 0.8554 | F1: 0.8518
  > XGBoost      | Prec: 0.8351 | Rec: 0.8405 | F1: 0.8349
  > Deeper_MLP   | Prec: 0.8674 | Rec: 0.8629 | F1: 0.8608
  > Generating aggregated plots for GS-BRCA...

==================== GS-LGG ====================
  > Fold 0...
  > Fold 1...
  > Fold 2...
  > Fold 3...
  > Fold 4...

GS-LGG Classifier Ablation Results (Averaged across folds):
  > Base_MLP     | Prec: 0.9850 | Rec: 0.9838 | F1: 0.9839
  > SVM          | Prec: 0.9660 | Rec: 0.9637 | F1: 0.9625
  > XGBoost      | Prec: 0.9661 | Rec: 0.9637 | F1: 0.9634
  > Deeper_MLP   | Prec: 0.9646 | Rec: 0.9635 | F1: 0.9634
  > Generating aggregated plots for GS-LGG...

==================== GS-OV ====================
  > Fold 0...
  > Fold